# Test con Ollama

Resetear la conversación sin perder el contexto de los archivos:
pythonhistorial = historial[:1]  # deja solo el system prompt

In [ ]:
# Esta celda contiene un comando de terminal (no Python) para instalar Ollama.
# Ollama es un programa que permite correr modelos de lenguaje (LLMs) de forma local
# en tu propio ordenador, sin necesidad de internet ni de pagar por una API.
#
# Para ejecutar este comando: abre una terminal y pégalo ahí,
# O activa esta celda en el notebook (funciona en JupyterLab con el magic ! al inicio).
#
# El comando descarga y ejecuta el script de instalación oficial de Ollama:
# -fsSL → opciones de curl: -f=falla silenciosamente en errores, -s=modo silencioso,
#          -S=muestra errores, -L=sigue redirecciones
# | sh → pasa la salida del curl directamente a la shell para ejecutarla
#Installar Ollama
curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Comando de terminal para descargar el modelo qwen2.5:14b con Ollama.
# "ollama pull" descarga el modelo desde el repositorio de Ollama a tu ordenador.
# qwen2.5:14b es un modelo de 14 mil millones de parámetros desarrollado por Alibaba.
# El sufijo :14b indica el tamaño (14 Billion parameters = 14B parámetros).
# Este modelo ocupa aproximadamente 9 GB en disco y funciona bien con texto en ruso.
# Solo hace falta ejecutarlo UNA VEZ — después el modelo queda guardado localmente.
#
# Para ejecutarlo: abre una terminal y escribe el comando, o usa "!" antes en Jupyter.
#Para datos
ollama pull qwen2.5:14b        # (14B, ~9GB)

In [1]:
# ollama es la librería de Python para comunicarse con el servidor Ollama local.
# Permite enviar texto al LLM y recibir respuestas, todo sin internet.
import ollama

# os da acceso a funciones del sistema operativo (variables de entorno, etc.).
import os

# Path para trabajar con rutas de archivos de forma cómoda.
from pathlib import Path

# Nota: si necesitas más "memoria" de contexto (para enviar archivos más largos al LLM),
# puedes iniciar Ollama desde la terminal con: ollama run qwen2.5:14b --num-ctx 32768
# num-ctx define cuántos tokens de contexto puede manejar el modelo en una sola conversación.

# Definimos la carpeta que contiene los archivos de chat de Jabber 2020.
carpeta = Path('../data_Vruto/ContiLeaks/raw/Conti Chat Logs 2020/Conti Chat Logs 2020')

def leer_archivos(carpeta: Path, extensiones=[".txt", ".csv", ".log", ".json"]) -> str:
    """
    Lee todos los archivos de una carpeta con las extensiones indicadas
    y los concatena en un solo texto para enviarlo al LLM como contexto.

    Parámetros:
        carpeta (Path): La carpeta donde buscar archivos.
        extensiones (list): Lista de extensiones a leer (por defecto: .txt, .csv, .log, .json)

    Devuelve:
        str: Un texto con el contenido de todos los archivos, separados por cabeceras.
    """
    contenido = []
    for archivo in carpeta.iterdir():
        # Solo leemos archivos con las extensiones especificadas.
        if archivo.suffix in extensiones:
            # Leemos el contenido del archivo. errors="ignore" descarta caracteres
            # que no se puedan leer en UTF-8 sin generar error.
            texto = archivo.read_text(encoding="utf-8", errors="ignore")

            # Añadimos el nombre del archivo como cabecera (###) y limitamos
            # cada archivo a 3000 caracteres para no exceder el límite de tokens del LLM.
            contenido.append(f"### {archivo.name}\n{texto[:3000]}")  # límite por archivo

    # Unimos todos los textos separados por una línea en blanco.
    return "\n\n".join(contenido)

# Leemos todos los archivos JSON de la carpeta y los concatenamos en un string.
contexto = leer_archivos(carpeta)

# Inicializamos el historial de conversación.
# El "historial" es una lista de mensajes que le enviamos al LLM para que recuerde
# lo que se ha dicho antes. El primero siempre es el "system prompt", que define
# el comportamiento del asistente y proporciona el contexto de los archivos.
historial = [
    {
        "role": "system",
        "content": f"Eres un asistente de análisis de datos. Responde en español. Aquí tienes los archivos:\n\n{contexto}"
    }
]

# Hacemos la primera consulta al LLM: le pedimos una descripción general de los datos.
# Aquí enviamos el contexto directamente en el mensaje de usuario (sin usar el historial),
# lo que es una alternativa más simple para consultas únicas.
respuesta = ollama.chat(
    model="qwen2.5:14b",
    messages=[
        {
            "role": "system",
            "content": "Eres un asistente de análisis de datos. Responde siempre en español."
        },
        {
            "role": "user",
            # Incluimos los archivos como parte del mensaje de usuario, para que el LLM
            # los tenga disponibles al responder.
            "content": f"Aquí tienes el contenido de mis archivos:\n\n{contexto}\n\n¿Qué puedes decirme sobre estos datos?"
        }
    ]
)

# Mostramos la respuesta del LLM.
# respuesta.message.content es el texto generado por el modelo.
print(respuesta.message.content)

Estos archivos contienen registros de chats o comunicaciones internas relacionadas con la creación y distribución de software malicioso, probablemente para fines de fraude bancario o robo de identidad. Algunos puntos clave:

1. Hay referencias a "builds" (versiones) de software que parecen ser modificados para evadir sistemas de seguridad.

2. Se mencionan productos como Chrome y Cloud, sugiriendo que el malware está diseñado para infectar navegadores web o servicios en la nube.

3. Hay conversaciones sobre cómo detectar si una máquina ya está "contaminada" (infectada) o no antes de instalar nuevo software malicioso.

4. Se habla de crear nuevas versiones ("proliferas") del malware para evadir detección.

5. Existe comunicación entre distintas personas que parecen estar involucradas en el desarrollo y distribución del malware.

6. Hay menciones a "checkers" (verificadores) para determinar si un sistema está infectado o no.

7. Se habla de la necesidad de crear versiones nuevas con las 

In [2]:
# Definimos una función para hacer consultas al LLM de forma conversacional.
# La clave aquí es que mantenemos el "historial": cada pregunta y respuesta se acumula,
# así el LLM recuerda lo que se habló antes en la misma sesión.
def preguntar(pregunta: str):
    """
    Envía una pregunta al LLM y muestra su respuesta.
    La función mantiene el historial de conversación en la variable global 'historial',
    lo que permite hacer preguntas de seguimiento que refieren a respuestas anteriores.

    Parámetros:
        pregunta (str): El texto de la pregunta que quieres hacer.

    No devuelve nada: imprime la respuesta directamente.
    """
    # Añadimos la pregunta del usuario al historial antes de enviarla.
    # role='user' indica que este mensaje viene del usuario (no del asistente ni del sistema).
    historial.append({"role": "user", "content": pregunta})

    # Enviamos el historial completo al LLM. Al incluir todos los mensajes anteriores,
    # el modelo puede responder teniendo en cuenta el contexto de la conversación.
    # num_ctx: 32768 → ventana de contexto ampliada para poder incluir todos los archivos.
    respuesta = ollama.chat(
        model="qwen2.5:14b",
        options={"num_ctx": 32768},
        messages=historial
    )

    # Extraemos el texto de la respuesta del LLM.
    mensaje = respuesta.message.content

    # Añadimos la respuesta al historial también, con role='assistant'.
    # Así, la próxima vez que preguntemos, el LLM recordará lo que ya respondió.
    historial.append({"role": "assistant", "content": mensaje})

    # Imprimimos la respuesta para que sea visible en el notebook.
    print(mensaje)

# --- Primera consulta de ejemplo ---
# Preguntamos al LLM en qué idioma están escritos los mensajes de los chats.
# El LLM tiene el contexto de los archivos desde el system prompt definido más arriba.
preguntar("¿En qué lenguas setán los textos de los mensajes?")

La mayoría de los textos en los mensajes parecen estar en ruso, pero con algunos elementos que podrían ser inglés o un código especial. Específicamente:

1. La gran parte del contenido está en ruso.
2. Hay algunas palabras o frases en inglés (como "привет", "hello").
3. Algunos de los mensajes contienen texto que parece estar encriptado u oculto (por ejemplo, secuencias como "AAAAAOMSgoOYPQ3VMEEoHObJQoWGHpR6ARHOKiVFDRVF7AZqIgAAAGgAdAB0AHAA").

En resumen, la mayoría de los mensajes están principalmente en ruso, pero con mezclas o elementos adicionales que no son claramente identificables como un único idioma estándar.


In [3]:
# Segunda consulta: preguntamos si el LLM detecta diferentes roles en los chats.
# Como el historial ya contiene el system prompt con los archivos y la respuesta
# anterior sobre idiomas, el LLM puede responder con más contexto.
# Esta pregunta es especialmente interesante porque sirve para comparar
# la respuesta "intuitiva" del LLM con el análisis sistemático de los notebooks 02 y 03.
preguntar("¿Creees que hay diferentes roles en los chats?")

Basado en el contenido de los mensajes, parece haber varios roles o papeles distintos entre las personas que participan en estos chats. Algunas observaciones sobre posibles roles:

1. Líderes/directores: Personas como "target" y "stern" que dan instrucciones y toman decisiones importantes.

2. Desarrolladores/programadores: Se mencionan tareas técnicas relacionadas con codificación, pruebas y desarrollo de software (ejemplo: discusiones sobre código fuente y pruebas).

3. Operativos/administrativos: Personas que coordinan acciones operativas como preparar builds o probar sistemas.

4. Soporte técnico: Algunas conversaciones indican resolución de problemas técnicos entre los miembros del equipo.

5. Comunicadores: Individuos que reportan el estado de las cosas y la comunicación interna (ejemplo: notificar sobre cambios en pruebas).

6. Recursos: Personas que proporcionan materiales o recursos necesarios para el funcionamiento del sistema (ejemplo: compartir archivos o claves).

Sin emba

In [4]:
preguntar("Dame algun ejemplo de esas decisiones importantes (en español) que toman target o stern")

Basado en los registros proporcionados, aquí hay algunos ejemplos de decisiones importantes que toman Target y Stern:

1. Decisión sobre qué hacer con un posible problema técnico:
   [30.06.2020 16:00:54] <target> знать что пошло не так
   [30.06.2020 16:00:56] <target> мы же обсуждали
   [30.06.2020 16:01:02] <target> появился файл в папке
   [30.06.2020 16:01:03] <target> скачанные
   [30.06.2020 16:01:05] <target> значит все ок
   [30.06.2020 16:01:06] <target> не появилс
   [30.06.2020 16:01:09] <target> значит хром порезал
   [30.06.2020 16:01:26] <target> скрипт
   
   En español:
   - Si aparece el archivo en la carpeta Descargas, significa que todo está bien.
   - Si no apareció, significa que Chrome lo cortó.

2. Tomar decisiones sobre qué hacer con las pruebas y los posibles problemas detectados: 
   [14:59:45] <strix> пока никак -- пробую сделать, так чтобы алерты заработали. Пока не получается, и не знаю, получится ли в итоге.
   
   En español:
   - Aún no se puede decir n